# Ride-Hailing Method Comparison (July 2023)

"
"This notebook compares day-to-day ridership movement across four NYC ride-hailing services for **July 2023**:
"
"- FHV
- HVFHV
- Green Taxi
- Yellow Taxi

"
"It loads `2023_07_*_hourly_trip_counts.csv` from `data/01-interim/TLC_ridehail`, builds daily totals, runs simple statistical comparisons, and saves processed outputs to `data/02-processed/TLC_ridehail`.


In [ ]:
import json
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
from scipy.stats import friedmanchisquare, pearsonr, ttest_rel

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 140)


In [ ]:
# Paths and settings
INPUT_DIR = Path('data/01-interim/TLC_ridehail')
OUTPUT_DIR = Path('data/02-processed/TLC_ridehail')
TAXI_ZONES_PATH = Path('data/02-processed/map_files/NYC_Taxi_Zones.geojson')

MONTH_TAG = '2023_07'
JULY_START = pd.Timestamp('2023-07-01 00:00:00')
JULY_END_EXCLUSIVE = pd.Timestamp('2023-08-01 00:00:00')

# Set to False if you want all NYC zones instead of Manhattan only
MANHATTAN_ONLY = True

SERVICE_FILES = {
    'fhv': f'{MONTH_TAG}_fhv_hourly_trip_counts.csv',
    'hvfhv': f'{MONTH_TAG}_hvfhv_hourly_trip_counts.csv',
    'green_taxi': f'{MONTH_TAG}_green_taxi_hourly_trip_counts.csv',
    'yellow_taxi': f'{MONTH_TAG}_yellow_taxi_hourly_trip_counts.csv',
}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR


In [ ]:
def load_manhattan_location_ids(geojson_path: Path) -> set[int]:
    with open(geojson_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    ids = set()
    for feature in data.get('features', []):
        props = feature.get('properties', {})
        if str(props.get('borough', '')).strip().lower() != 'manhattan':
            continue
        raw_id = props.get('locationid', props.get('LocationID'))
        if raw_id is None:
            continue
        try:
            ids.add(int(raw_id))
        except (TypeError, ValueError):
            pass
    return ids


def load_hourly_csv(csv_path: Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    df['pickup_hour'] = pd.to_datetime(df['pickup_hour'], errors='coerce')
    df['pickup_location'] = pd.to_numeric(df['pickup_location'], errors='coerce').astype('Int64')
    df['trip_count'] = pd.to_numeric(df['trip_count'], errors='coerce')

    df = df.dropna(subset=['pickup_hour', 'pickup_location', 'trip_count'])
    df = df[df['pickup_hour'] >= JULY_START]
    df = df[df['pickup_hour'] < JULY_END_EXCLUSIVE]
    return df


def benjamini_hochberg(p_values: np.ndarray) -> np.ndarray:
    p_values = np.asarray(p_values, dtype=float)
    n = len(p_values)
    order = np.argsort(p_values)
    ranked = p_values[order]

    adjusted_ranked = np.empty(n, dtype=float)
    prev = 1.0
    for i in range(n - 1, -1, -1):
        rank = i + 1
        val = ranked[i] * n / rank
        prev = min(prev, val)
        adjusted_ranked[i] = prev

    adjusted = np.empty(n, dtype=float)
    adjusted[order] = np.clip(adjusted_ranked, 0, 1)
    return adjusted


In [ ]:
# Load hourly data for each service
manhattan_ids = load_manhattan_location_ids(TAXI_ZONES_PATH) if MANHATTAN_ONLY else None

hourly = {}
for service, filename in SERVICE_FILES.items():
    df = load_hourly_csv(INPUT_DIR / filename)
    if MANHATTAN_ONLY:
        df = df[df['pickup_location'].isin(manhattan_ids)]
    hourly[service] = df

summary = pd.DataFrame({
    'service': list(hourly.keys()),
    'rows': [len(hourly[s]) for s in hourly],
    'total_trips': [int(hourly[s]['trip_count'].sum()) for s in hourly],
    'first_hour': [hourly[s]['pickup_hour'].min() for s in hourly],
    'last_hour': [hourly[s]['pickup_hour'].max() for s in hourly],
})
summary


In [ ]:
# Save filtered hourly files (same style as 01-interim folder, but month-specific names)
for service, df in hourly.items():
    out_path = OUTPUT_DIR / f'{MONTH_TAG}_{service}_hourly_trip_counts.csv'
    df.sort_values(['pickup_hour', 'pickup_location']).to_csv(out_path, index=False)

all_hourly = (
    pd.concat(hourly.values(), ignore_index=True)
    .groupby(['pickup_hour', 'pickup_location'], as_index=False)['trip_count']
    .sum()
    .sort_values(['pickup_hour', 'pickup_location'])
)
all_hourly.to_csv(OUTPUT_DIR / f'{MONTH_TAG}_all_ridehail_hourly_trip_counts.csv', index=False)

print('Saved hourly outputs to', OUTPUT_DIR)
all_hourly.head()


In [ ]:
# Build daily totals by service
daily_parts = []
for service, df in hourly.items():
    d = (
        df.groupby(df['pickup_hour'].dt.normalize(), as_index=False)['trip_count']
        .sum()
        .rename(columns={'pickup_hour': 'date', 'trip_count': service})
    )
    daily_parts.append(d)
    d.rename(columns={service: 'trip_count'}).to_csv(
        OUTPUT_DIR / f'{MONTH_TAG}_{service}_daily_trip_counts.csv', index=False
    )

daily = daily_parts[0]
for d in daily_parts[1:]:
    daily = daily.merge(d, on='date', how='outer')

daily = daily.sort_values('date').reset_index(drop=True)

all_daily = (
    daily[['date'] + list(SERVICE_FILES.keys())]
    .set_index('date')
    .sum(axis=1)
    .rename('trip_count')
    .reset_index()
)
all_daily.to_csv(OUTPUT_DIR / f'{MONTH_TAG}_all_ridehail_daily_trip_counts.csv', index=False)

daily.head()


## Why We Use Log Day-Over-Day Changes

We compare services using:

$$ \log\left(rac{	ext{today trips}}{	ext{yesterday trips}}ight) $$

Why:
- It measures **proportional** change (good when service sizes differ a lot).
- 0 means no change, positive means increase, negative means decrease.
- It works naturally with multiplicative behavior (common in demand data).


In [ ]:
# Keep only dates where all services are present
services = list(SERVICE_FILES.keys())
complete = daily.dropna(subset=services).copy()

ratios = np.log(complete[services] / complete[services].shift(1))
ratios = ratios.replace([np.inf, -np.inf], np.nan).dropna()

ratio_with_date = ratios.copy()
ratio_with_date.insert(0, 'date', complete['date'].iloc[1:].values)
ratio_with_date.head()


In [ ]:
# 1) Friedman repeated-measures test (overall difference across all services)
friedman = friedmanchisquare(*(ratios[s].values for s in services))
overall_test = pd.DataFrame([{
    'test': 'friedman_repeated_measures',
    'n_days': len(ratios),
    'statistic': float(friedman.statistic),
    'p_value': float(friedman.pvalue),
}])
overall_test


In [ ]:
# 2) Pairwise paired t-tests
pair_rows = []
for a, b in combinations(services, 2):
    t_res = ttest_rel(ratios[a], ratios[b], nan_policy='omit')
    diff = ratios[a] - ratios[b]
    pair_rows.append({
        'service_a': a,
        'service_b': b,
        'n_days': int(diff.notna().sum()),
        'mean_diff_log_ratio': float(diff.mean()),
        't_statistic': float(t_res.statistic),
        'p_value': float(t_res.pvalue),
    })

pairwise_ttests = pd.DataFrame(pair_rows)
pairwise_ttests['p_value_fdr_bh'] = benjamini_hochberg(pairwise_ttests['p_value'].to_numpy())
pairwise_ttests.sort_values('p_value_fdr_bh')


In [ ]:
# 3) Pairwise Pearson correlations (do they move together day-to-day?)
corr_rows = []
for a, b in combinations(services, 2):
    r, p = pearsonr(ratios[a], ratios[b])
    corr_rows.append({
        'service_a': a,
        'service_b': b,
        'n_days': len(ratios),
        'pearson_r': float(r),
        'p_value': float(p),
    })

pairwise_correlations = pd.DataFrame(corr_rows)
pairwise_correlations['p_value_fdr_bh'] = benjamini_hochberg(pairwise_correlations['p_value'].to_numpy())
pairwise_correlations.sort_values('pearson_r', ascending=False)


In [ ]:
ratio_summary = ratios.describe().T.reset_index().rename(columns={'index': 'service'})

# Save processed analysis tables (month-specific filenames)
daily.to_csv(OUTPUT_DIR / f'{MONTH_TAG}_daily_totals.csv', index=False)
ratio_with_date.to_csv(OUTPUT_DIR / f'{MONTH_TAG}_daily_log_ratios.csv', index=False)
overall_test.to_csv(OUTPUT_DIR / f'{MONTH_TAG}_overall_test.csv', index=False)
pairwise_ttests.to_csv(OUTPUT_DIR / f'{MONTH_TAG}_pairwise_ttests.csv', index=False)
pairwise_correlations.to_csv(OUTPUT_DIR / f'{MONTH_TAG}_pairwise_correlations.csv', index=False)
ratio_summary.to_csv(OUTPUT_DIR / f'{MONTH_TAG}_ratio_summary.csv', index=False)

print('Saved processed analysis tables to', OUTPUT_DIR)
ratio_summary


## Interpreting the Results

- `*_overall_test.csv` (Friedman):
  - Null hypothesis: all services have the same day-to-day log-change distribution.
  - Small p-value: at least one service behaves differently.

- `*_pairwise_ttests.csv`:
  - Null hypothesis for each pair: mean daily log-change difference is 0.
  - Use `p_value_fdr_bh` for significance (multiple-testing corrected).

- `*_pairwise_correlations.csv`:
  - Null hypothesis for each pair: correlation is 0.
  - Higher `pearson_r` means stronger synchronized movement.
  - Use `p_value_fdr_bh` for significance.
